In [ ]:



from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import os
import yaml
import random

In [ ]:
# Output directory
BASE_DIR = "datasets/yolo"
IMG_TRAIN = os.path.join(BASE_DIR, "images/train")
IMG_VAL = os.path.join(BASE_DIR, "images/val")
LBL_TRAIN = os.path.join(BASE_DIR, "labels/train")
LBL_VAL = os.path.join(BASE_DIR, "labels/val")


for d in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
os.makedirs(d, exist_ok=True)


# Selected 25 COCO classes (name: original COCO id)
COCO_CLASSES = {
"person": 0,
"car": 2,
"truck": 7,
"bus": 5,
"motorcycle": 3,
"bicycle": 1,
"airplane": 4,
"traffic light": 9,
"stop sign": 11,
"bench": 13,
"dog": 16,
"cat": 15,
"horse": 17,
"bird": 14,
"cow": 19,
"elephant": 20,
"bottle": 39,
"cup": 41,
"bowl": 45,
"pizza": 59,
"cake": 61,
"chair": 56,
"couch": 57,
"bed": 59,
"potted plant": 58
}


# Remap to YOLO class IDs (0–24)
CLASS_ID_MAP = {v: i for i, v in enumerate(COCO_CLASSES.values())}
CLASS_NAMES = list(COCO_CLASSES.keys())


IMAGES_PER_CLASS = 100
TRAIN_SPLIT = 0.7

In [ ]:
print("Loading COCO dataset (streaming mode)...")
dataset = load_dataset("detection-datasets/coco", "2017", split="train", streaming=True)

In [ ]:


def coco_to_yolo_bbox(bbox, img_w, img_h):
# COCO bbox: [x_min, y_min, width, height]
x, y, w, h = bbox
x_center = (x + w / 2) / img_w
y_center = (y + h / 2) / img_h
return x_center, y_center, w / img_w, h / img_h

In [ ]:
class_counts = {cid: 0 for cid in CLASS_ID_MAP.keys()}
collected = []


print("Collecting images...")
for sample in tqdm(dataset):
anns = sample["annotations"]
valid_anns = []


for ann in anns:
cid = ann["category_id"]
if cid in CLASS_ID_MAP and class_counts[cid] < IMAGES_PER_CLASS:
valid_anns.append(ann)


if not valid_anns:
continue


img = Image.open(sample["image"]).convert("RGB")
img_w, img_h = img.size


collected.append((img, valid_anns))


for ann in valid_anns:
class_counts[ann["category_id"]] += 1


if all(v >= IMAGES_PER_CLASS for v in class_counts.values()):
break


print("Class counts:")
for k, v in class_counts.items():
print(k, v)

In [ ]:


random.shuffle(collected)
split_idx = int(len(collected) * TRAIN_SPLIT)
train_data = collected[:split_idx]
val_data = collected[split_idx:]

In [ ]:


def save_split(data, img_dir, lbl_dir, prefix):
    for idx, (img, anns) in enumerate(tqdm(data)):
        img_name = f"{prefix}_{idx}.jpg"
        lbl_name = f"{prefix}_{idx}.txt"


img_path = os.path.join(img_dir, img_name)
lbl_path = os.path.join(lbl_dir, lbl_name)


img.save(img_path)
w, h = img.size


with open(lbl_path, "w") as f:
    for ann in anns:
        cid = CLASS_ID_MAP[ann["category_id"]]
        x, y, bw, bh = coco_to_yolo_bbox(ann["bbox"], w, h)
        f.write(f"{cid} {x} {y} {bw} {bh}\n")


print("Saving training data...")
save_split(train_data, IMG_TRAIN, LBL_TRAIN, "train")


print("Saving validation data...")
save_split(val_data, IMG_VAL, LBL_VAL, "val")

In [ ]:


data_yaml = {
"path": BASE_DIR,
"train": "images/train",
"val": "images/val",
"nc": len(CLASS_NAMES),
"names": CLASS_NAMES
}


with open(os.path.join(BASE_DIR, "data.yaml"), "w") as f:
yaml.dump(data_yaml, f)


print("data.yaml created")

In [ ]:


print("Sanity check:")
print("Train images:", len(os.listdir(IMG_TRAIN)))
print("Val images:", len(os.listdir(IMG_VAL)))
print("Train labels:", len(os.listdir(LBL_TRAIN)))
print("Val labels:", len(os.listdir(LBL_VAL)))


print("\nYOLO dataset preparation COMPLETE ")